In [ ]:
%pip install  langchain langchain-openai langchain-community langchain-chroma

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

# 1. Document parsing

In [4]:
loader = TextLoader("word.pdf")
documents = loader.load()

# 2. Chunking

Manually: 
chunks = []

for document in documents:
    chunk = ...
    chunks.append(chunk)

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)
print(type(chunks))

<class 'list'>


# 3. Embeddings

In [ ]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# 4. Store embeddings in vector database

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

# 5. Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# 6. Prompt

In [13]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}
""")

# 7. LLM

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

# 8. Query Pipeline

In [ ]:
def ask(question):

    # Similarity search
    docs = retriever.invoke(question) # here retriver object convert que to embeding and search in vector store and return top k docs

    # Combine retrieved chunks
    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    # Create prompt
    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    # Ask LLM
    response = llm.invoke(messages)

    return response.content


print(ask("What is the company's vacation policy?"))